# Original linear FDM

Standalone run on the supplied **SWG 256×256 geometry, 12 µm × 12 µm**. Edit the **Parameters** cell, then select **Kernel → Restart Kernel and Run All Cells**. Paths are relative to the repository root. Results are saved under `outputs/`.

No trained weights or external dataset are required. The input geometry is already included.


In [ ]:
%matplotlib inline
from pathlib import Path
import os
import sys

root_candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((p for p in root_candidates if (p / "plate_models.py").is_file()
             and (p / "data/swg_256.npz").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or its notebooks directory")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)

from pathlib import Path
import numpy as np
from fdm_solver import get_w_strain


## Parameters

Change the input file/key, physical domain size, physical parameters and output path here. Support maps use 1 for supports and 0 for suspended film. `width_um` sets **both** length and width of the square domain; independent unequal x/y extents are not exposed by these solvers. Changing the domain size scales every feature; change the input pixels to change individual teeth or gaps.
This original solver fixes E, nu, h and pressure internally. Use `02_linear_plate.ipynb` for adjustable material and loading.


In [ ]:
structure_file = "data/swg_256.npz"
structure_key = "array"
width_um = 12.0
output_file = "outputs/swg_linear_original.npz"


## Calculate and save

`get_w_strain` takes width in **metres**, so the example converts µm to m. It returns dimensionless bending strain and coordinate arrays in metres; it does not return deflection. For the supplied SWG, the original bending-strain peak is approximately 0.230908%.

This historical function fixes E=167.3 GPa, nu=0.19, h=1 nm and pressure=−33.87 Pa inside `fdm_solver.py`, in the local variables `E`, `nu`, `h` and `q_pressure`. Its API does not accept these parameters. Use the configurable linear solver below to change material or pressure through arguments.

In [ ]:
with np.load(structure_file) as data:
    support = data[structure_key]
strain_fraction, x_m, y_m = get_w_strain(support, width=width_um * 1e-6)
Path(output_file).parent.mkdir(parents=True, exist_ok=True)
np.savez_compressed(output_file, support_map=support, strain_fraction=strain_fraction,
                    X_coord=x_m, Y_coord=y_m, width_m=width_um * 1e-6)
print("Peak bending strain (%):", strain_fraction.max() * 100)


## Inspect the result

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].imshow(support, origin="lower", extent=[0, width_um, 0, width_um])
axes[0].set_title("SWG supports")
im = axes[1].imshow(strain_fraction * 100, origin="lower", extent=[0, width_um, 0, width_um])
axes[1].set_title("Strain (%)")
fig.colorbar(im, ax=axes[1])
for ax in axes:
    ax.set_xlabel("x (um)")
    ax.set_ylabel("y (um)")
plt.show()
